# Optimización de turnos y cobertura: optimización

Consulta la [metodología](../docs/metodologia.md) y ejecuta las etapas en orden. Los comentarios se calculan con los resultados guardados.

In [1]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("workforce_scheduling")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre este notebook desde su proyecto independiente.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from workforce_scheduling.alerts import configurar_notebook, mostrar_alertas
from IPython.display import display, Markdown, Image
configurar_notebook()


In [2]:
from workforce_scheduling.analysis import run_stage
import pandas as pd
metrics,comment=run_stage(PROJECT_ROOT,"modelado")
display(Markdown(comment))
scalars={k:v for k,v in metrics.items() if not isinstance(v,(dict,list))}
if scalars:display(pd.DataFrame(list(scalars.items()),columns=["Indicador","Valor"]))
if "candidatos" in metrics:display(pd.DataFrame(metrics["candidatos"]))
for filename in ["eda.png","evaluacion.png"]:
    path=PROJECT_ROOT/"reports/modelado"/filename
    if path.exists():display(Image(filename=str(path)))
for filename in ["estimaciones.csv","metricas_test.csv","horas_por_empleado.csv"]:
    path=PROJECT_ROOT/"reports/modelado"/filename
    if path.exists():display(pd.read_csv(path).head(20))

# Optimización de turnos y cobertura: modelado

MILP binario minimiza coste pagado sujeto a cobertura, un turno diario por empleado, disponibilidad, máximo **40 horas semanales** y **4 días consecutivos**. Cada turno dura 8 horas y contiene dos pausas de 30 minutos sin cobertura. El solver **terminó dentro de la tolerancia configurada**; gap relativo **0.00%**, coste **5,904.00 unidades**. Se audita la solución fuera del solver. La certificación depende de la tolerancia y los patrones disponibles; no se afirma un óptimo exacto si el gap no es cero. El escenario de equilibrio terminó **por límite de tiempo, con plan factible y sin certificar optimalidad**, con gap de su objetivo **22.00%**.


,Indicador,Valor
0,solver_status,0
1,solver_message,Optimization terminated successfully. (HiGHS S...
2,gap_relativo,0.0
3,cota_inferior_coste,5904.0
4,objetivo_coste,5904.0
5,variables_binarias,3360
6,tolerancia_gap,0.01
7,restricciones_laborales_cumplidas,True
8,cobertura_completa,True
9,deficit_agente_intervalos,0.0


In [3]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/04_modelado.json")
from workforce_scheduling.informe import generar_informe
generar_informe(PROJECT_ROOT)
display(Markdown("[Abrir informe integral](../reports/informe_final/informe_final.md)"))

Tipo,Veces,Qué significa,Qué revisar,Detalle original
SolverLimitWarning,1,"El solver entregó un plan factible, pero terminó sin cerrar el gap solicitado.",Revisar el gap y el plan auditado; aumentar el límite o estudiar otra formulación si se necesita certificar optimalidad.,"Ver detalle y origenEl solver del escenario de equilibrio alcanzó su límite de tiempo. Plan factible auditado; gap del objetivo alternativo 22.00%, sin certificar optimalidad.D:\portafolio\Analisis\Optimization\WorkforceScheduling\src\workforce_scheduling\balance.py:52"


[Abrir informe integral](../reports/informe_final/informe_final.md)